# Estimación espectral no-paramétrica: LIGO

**22.46 Procesamiento Adaptativo de Señales Aleatorias — Trabajo Práctico 1**

---

# Grupo N° 5

| Integrante | Legajo |
|---|---|
| Albertine Gjøs | 69531 |
| Marek Myrebøe-Engels | 69476 |

**Fecha de entrega:** 08/09

---

> **Cómo se entrega.** Este notebook **es** el informe. Completen la portada de arriba,
> resuelvan cada inciso en su sección correspondiente **sin alterar el orden ni la numeración**,
> y dejen en cada celda de respuesta la justificación escrita: las decisiones de parametrización
> son el contenido evaluado, no un detalle de implementación.
>
> Entreguen el notebook **ejecutado de principio a fin y guardado con las salidas visibles**,
> con el nombre `TP1_LIGO_Grupo<N>.ipynb`. Usen rutas relativas a `data/` y **no incluyan los archivos de datos**
> en la entrega. El notebook debe correr de arriba a abajo sin intervención manual.

## Presentación

En este trabajo deberán analizar la onda gravitatoria **GW150914**, la primera detección directa de la historia. Fue registrada por el proyecto **LIGO** (Laser Interferometer Gravitational-Wave Observatory) el 14 de septiembre de 2015, y corresponde a la fusión de dos agujeros negros de 36 y 29 masas solares a unos 1300 millones de años luz.

El interés para esta materia es instrumental: **la señal llega enterrada bajo un ruido que la supera ampliamente, sobre un instrumento cuyo espectro abarca varias décadas de amplitud**. Todo el trabajo consiste en estimar bien ese ruido y usar esa estimación para recuperar la señal.

### Los detectores

Cada detector es un interferómetro de Michelson con brazos de 4 km y cavidades Fabry-Perot. Lo que mide es la **deformación** relativa del espacio, o *strain*:

$$h(t) = \frac{\Delta L(t)}{L}$$

Para GW150914 el pico fue $h \approx 10^{-21}$: un cambio de longitud de brazo de $4 \times 10^{-18}\,\mathrm{m}$, cuatro órdenes de magnitud por debajo del diámetro de un protón. Tengan presente esa cifra cuando evalúen qué tan cerca del ruido está lo que buscan.

Hay dos detectores, en Hanford (Washington) y en Livingston (Louisiana), separados por $d = 3002\,\mathrm{km}$. Como las ondas gravitatorias viajan a la velocidad de la luz, el retardo entre ambos está acotado por

$$\Delta t_{\max} = \frac{d}{c} \approx 10\,\mathrm{ms}$$

En la notación de GWOSC, **H1** es Hanford y **L1** es Livingston.

### El ruido de fondo

La sensibilidad está limitada por varias contribuciones simultáneas, cada una dominante en una región distinta del espectro:

<img src="ligo_noise.png" alt="LIGO Noise" width="400px"/>

**Atención: es una curva de diseño esquemática, no una medición.** Indica qué mecanismos limitan la sensibilidad y dónde domina cada uno, pero los niveles absolutos no corresponden al instrumento con el que van a trabajar.

A esto se suman **componentes tonales**: modos mecánicos de las suspensiones, líneas de red y sus armónicas, líneas inyectadas para calibración, resonancias de la óptica.

### El set de datos

Los datos son públicos y están en **GWOSC** (Gravitational Wave Open Science Center): [gwosc.org/events/GW150914](https://gwosc.org/events/GW150914/)

Descarguen los datos de *strain* a **4096 Hz** para H1 y L1. Para cada detector hay dos archivos HDF5:

| Archivo | Duración | Uso previsto |
|---|---|---|
| Ventana corta centrada en el evento | 32 s | Whitening y DOA (puntos 3 y 4) |
| Ventana larga | 4096 s | Espectro de ruido (punto 2) |

Los nombres siguen el patrón `<D>-<D1>_LOSC_4_V2-<GPS_inicio>-<duración>.hdf5`, por ejemplo `H-H1_LOSC_4_V2-1126259446-32.hdf5`.

El evento es una señal de tipo **chirp** que ocurre cerca del tiempo GPS $1126259462{,}4\,\mathrm{s}$ y dura aproximadamente $200\,\mathrm{ms}$. Puede verse y escucharse en [este video](https://www.youtube.com/watch?v=QyDcTbR-kEA).

### Notas de implementación

- **Grafiquen siempre en escala log-log.** Con este rango dinámico, un gráfico lineal no muestra nada.
- Es habitual reportar la **amplitud** espectral $\sqrt{S_x(f)}$ en $1/\sqrt{\mathrm{Hz}}$ en lugar de la potencia. Elijan una convención y sean consistentes.
- Verifiquen la **normalización** de sus estimadores: una PSD mal escalada por el factor de la ventana pasa desapercibida en escala logarítmica pero arruina el whitening.
- Usen `sosfiltfilt` en lugar de `lfilter`: un corrimiento de fase entre H1 y L1 destruiría la estimación de retardo del punto 4.
- Documenten **todas** las decisiones de parametrización ($N$, $L$, $K$, ventana, solapamiento, banda) y su justificación.

> **Sobre el Anexo.** Al final de este notebook hay un anexo con valores publicados por la colaboración LIGO, dentro de una sección plegada. **Ábranlo recién después de haber producido sus propios resultados**, y usen la comparación como material de discusión.

## 0. Preparación

Celda de configuración común. Ajusten `DATA` si sus archivos están en otro lado.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import h5py                              # lectura de los HDF5 de GWOSC
from pathlib import Path
from scipy.signal import welch, periodogram, correlate
from scipy.signal import butter, sosfiltfilt
from scipy.signal.windows import tukey, hann
from scipy.interpolate import interp1d
from scipy.io import wavfile             # exportación del audio

DATA = Path("data")                      # carpeta con los .hdf5 descargados de GWOSC
FS = 4096                                # Hz
GPS_EVENTO = 1126259462.4                # s


def leer_strain(path):
    """Devuelve (strain, t0, dt) de un archivo de strain de GWOSC."""
    with h5py.File(path, "r") as f:
        strain = f["strain"]["Strain"][:]
        t0 = f["strain"]["Strain"].attrs["Xstart"]
        dt = f["strain"]["Strain"].attrs["Xspacing"]
    return strain, t0, dt


plt.rcParams["figure.figsize"] = (10, 4)

ModuleNotFoundError: No module named 'h5py'

---

## 1. Investigación preliminar

Investiguen cómo funcionan los detectores de LIGO y escriban un resumen breve que cubra los mecanismos físicos que limitan la sensibilidad y en qué banda domina cada uno, qué tipo de artefactos aparecen en los datos (líneas espectrales, transitorios, no estacionariedad) y qué rango dinámico esperan encontrar en la amplitud espectral.

> **Resumen.** *(completar)*

---

## 2. Estimación del espectro de ruido

Estimen el espectro de potencia del ruido de H1 y L1 usando las partes de la señal que **no** involucran el evento. Justifiquen qué tramo excluyeron.

### 2.a — Periodograma

Estímenlo con el **periodograma**. Seleccionen y justifiquen la cantidad de datos $N$.

> **Respuesta.** *(completar)*

### 2.b — Periodogram smoothing

Estímenlo con **periodogram smoothing**. Seleccionen y justifiquen $N$ y $L$. Iteren hasta obtener una buena estimación.

> **Respuesta.** *(completar)*

### 2.c — Periodogram averaging

Estímenlo con **periodogram averaging**. Seleccionen y justifiquen $N$, $K$ y $L$. Iteren hasta obtener una buena estimación.

> **Respuesta.** *(completar)*

### 2.d — Comparación de métodos

Determinen y justifiquen cuál de los tres produce la mejor estimación, discutiendo el compromiso entre resolución, sesgo y varianza. Incluyan en la discusión la elección de la **ventana**: dado el rango dinámico de este problema, comparen al menos una ventana con *taper* contra la rectangular.

> **Respuesta.** *(completar)*

### 2.e — H1 contra L1

Comparen las estimaciones de H1 y L1 entre sí: picos espectrales, artefactos, sesgo, varianza y cualquier otra cosa que les llame la atención. Atribuyan un origen a las componentes tonales que encuentren. ¿Cómo se compara lo que midieron con la curva esquemática de la presentación?

> **Respuesta.** *(completar)*

### 2.f — Estacionariedad

**Validen la estacionariedad** de las señales. Investiguen técnicas de validación y aplíquenlas. ¿Sobre qué horizonte temporal es razonable tratar el ruido como estacionario?

> **Respuesta.** *(completar)*

---

## 3. Whitening

Con la mejor estimación del punto 2, realicen el **whitening** de los 32 s alrededor del evento: en el dominio de la frecuencia, normalicen cada componente por la amplitud espectral del ruido. Referencia útil: [los tutoriales de GWOSC](https://gwosc.org/tutorials/).

> **Whitening.** *(completar)*

### 3.a — Filtrado y visualización

Filtren pasabanda en la región de mayor sensibilidad, justificando los límites a partir de sus estimaciones del punto 2, y grafiquen la señal en el tiempo para ambos detectores. Deberían ver el chirp.

> **Respuesta.** *(completar)*

### 3.b — Audio

Generen el audio. El resultado directo es difícil de escuchar: expliquen por qué y qué transformación adicional aplicaron.

> **Respuesta.** *(completar)*

---

## 4. Dirección de arribo (DOA)

Con las señales del punto anterior, estimen por **correlación cruzada** la diferencia de tiempo de arribo del evento entre H1 y L1.

### 4.a — Retardo

Reporten el retardo del pico y su signo: ¿qué detector recibió la señal primero?

> **Pista.** A 4096 Hz el intervalo entre muestras es de 244 µs y el retardo máximo posible es de 10 ms. Evalúen si la resolución de una muestra les alcanza.

> **Respuesta.** *(completar)*

### 4.b — Ángulo de cono

Calculen el **ángulo del cono de arribo** respecto de la línea que une ambos detectores, y comparen con el Anexo:

$$\cos\theta = \frac{c\,\Delta t}{d}$$

¿Por qué el resultado es un cono y no una dirección? ¿Qué haría falta para localizar la fuente en un punto?

> **Respuesta.** *(completar)*

---

## Anexo: valores de referencia

<details>
<summary><b>⚠️ No abrir antes de haber completado los puntos correspondientes — clic para desplegar</b></summary>

<br>

Sirve para **contrastar** sus resultados, no para orientarlos. Fuente: Abbott et al., *Observation of Gravitational Waves from a Binary Black Hole Merger*, Phys. Rev. Lett. 116, 061102 (2016).

**Evento** (puntos 2 y 3). GW150914 ocurrió el 14/09/2015 a las 09:50:45 UTC, GPS 1126259462,4. La señal barre de ~35 Hz a ~250 Hz en unos 0,2 s, completando unos 8 ciclos, con un *strain* pico de 1,0 &times; 10<sup>-21</sup>.

**Sensibilidad** (punto 2.e). En la corrida O1, la amplitud espectral de ruido de ambos detectores alcanzó un mínimo del orden de 10<sup>-23</sup> Hz<sup>-1/2</sup> entre ~100 y ~300 Hz, degradándose rápidamente por debajo de 20 Hz.

**Retardo y localización** (punto 4). L1 registró el evento 6,9 (+0,5 / −0,4) ms **antes** que H1. Con solo dos detectores, la región de credibilidad al 90 % abarcó unos 600 grados cuadrados.

</details>

---

## Extensiones opcionales

Para quienes quieran ir más allá: mostrar el evento en un **espectrograma**; verificar la hipótesis de **gaussianidad** del ruido, que es distinta de la de estacionariedad; usar las máscaras `DQmask` e `injmask` de los archivos para descartar tramos no utilizables e inyecciones de prueba; estimar el retardo con precisión **sub-muestral** y propagar la incertidumbre hasta $\theta$; explicar el signo del pico de correlación cruzada a partir de la orientación relativa de los brazos de ambos detectores.